# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


####  Run this cell to set up and start your interactive session.


In [7]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5


In [1]:

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, when, to_date
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 45155610-0be1-43f1-9228-e21b755a5da8
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session 45155610-0be1-43f1-9228-e21b755a5da8 to get into ready status...
Session 45155610-0be1-43f1-9228-e21b755a5da8 has been created.



# Read CSV from S3


In [2]:
# ---- Read CSV from S3 ----
s3_path = "s3://banking-collection-app-datalake-eu-west2/Bronze/"
raw_df = spark.read.option("header", True).csv(s3_path)




+-----------+---------------+--------------+------------+-------------+-----------+------------------+----------+--------+--------------+----------------+-----------------+------------------+------+--------------------+--------------------+--------------+----------+--------------------+
|Customer_ID|           Name|Account_Number|Account_Type|    Loan_Type|Loan_Amount|Outstanding_Amount|EMI_Amount|Due_Date|Payment_Status|Collection_Agent|Last_Payment_Date|Payment_Delay_Days|Region|      Contact_Number|               Email|Customer_Score|Risk_Level|Payment_On_Time_Flag|
+-----------+---------------+--------------+------------+-------------+-----------+------------------+----------+--------+--------------+----------------+-----------------+------------------+------+--------------------+--------------------+--------------+----------+--------------------+
|  CUST00000|Richard Mccarty|  81087F51-C9F|     Current|     Car Loan|   299783.0|         119495.19|   2597.72|    NULL|          Paid

### Transformation

transformed_df = (
    raw_df
    .withColumn("Loan_Amount", col("Loan_Amount").cast("double"))
    .withColumn("Outstanding_Amount", col("Outstanding_Amount").cast("double"))
    .withColumn("EMI_Amount", col("EMI_Amount").cast("double"))
    .withColumn("Due_Date", to_date(col("Due_Date"), "dd-MM-yyyy"))
    .withColumn("Last_Payment_Date", to_date(col("Last_Payment_Date"), "dd-MM-yyyy"))
    .withColumn("Payment_Delay_Days", col("Payment_Delay_Days").cast("int"))
    .withColumn(
        "Payment_On_Time_Flag",
        when(col("Payment_Delay_Days") == 0, True).otherwise(False)
    )
)
transformed_df.show(5)

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [4]:
# ---- Dimension Customer ----
dim_customer_df = transformed_df.select(
    "Customer_ID", "Name", "Region", "Contact_Number",
    "Email", "Customer_Score", "Risk_Level"
).dropDuplicates(["Customer_ID"])

dim_customer_df.write.mode("overwrite").parquet("s3://banking-collection-app-datalake-eu-west2/Silver/dim_customer/")

# ---- Dimension Account ----
dim_account_df = transformed_df.select(
    "Account_Number", "Account_Type", "Loan_Type"
).dropDuplicates(["Account_Number"])

dim_account_df.write.mode("overwrite").parquet("s3://banking-collection-app-datalake-eu-west2/Silver/dim_account/")

# ---- Fact Loan Payment ----
fact_loan_payment_df = transformed_df.select(
    "Customer_ID", "Account_Number", "Loan_Amount", "Outstanding_Amount",
    "EMI_Amount", "Due_Date", "Payment_Status", "Last_Payment_Date",
    "Payment_Delay_Days", "Payment_On_Time_Flag"
)

fact_loan_payment_df.write.mode("overwrite").parquet("s3://banking-collection-app-datalake-eu-west2/Silver/fact_loan_payment/")


In [3]:
output_path = "s3://banking-collection-app-datalake-eu-west2/Silver/"
transformed_df.write.mode("overwrite").parquet(output_path)